#Black-Box attack on CombinedDetector

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split , Dataset
import timm
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import torch.nn.functional as F   # need for F.cross_entropy
torch.manual_seed(42)              # reproducibility
np.random.seed(42)

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

PyTorch: 2.11.0+cu128
GPU: NVIDIA L4


##Setup and data loading

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
import os
import shutil

with zipfile.ZipFile('/content/drive/MyDrive/data_combined_8k.zip', 'r') as z:
    for member in z.namelist():
        fixed_path = member.replace('\\', '/')
        target = os.path.join('/content', fixed_path)
        if member.endswith('\\') or '.' not in os.path.basename(fixed_path):
            os.makedirs(target, exist_ok=True)
        else:
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with z.open(member) as src, open(target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

print("Real frames:", len(os.listdir('/content/data_combined_8k/real')))
print("Fake frames:", len(os.listdir('/content/data_combined_8k/fake')))

Mounted at /content/drive
Real frames: 8000
Fake frames: 8000


In [3]:
train_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = self.subset.dataset.loader(path)
        return self.transform(image), label


full_dataset = ImageFolder('/content/data_combined_8k', transform=None)
print("Classes:", full_dataset.classes)
print("Total:", len(full_dataset))

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_sub, val_sub, test_sub = random_split(full_dataset, [train_size, val_size, test_size])

train_set = TransformSubset(train_sub, train_transform)
val_set = TransformSubset(val_sub, test_transform)
test_set = TransformSubset(test_sub, test_transform)

train_loader = DataLoader(train_set, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set, batch_size=32, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")

Classes: ['fake', 'real']
Total: 16000
Train: 11200, Val: 2400, Test: 2400


##Defining CombinedDetector Architecture
(target model)

In [4]:
class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1),  # 6 channels: RGB magnitude + RGB phase
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

    def forward(self, x):
        fft = torch.fft.fft2(x)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))

        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)
        magnitude = (magnitude - magnitude.mean(dim=(-2, -1), keepdim=True)) / \
                    (magnitude.std(dim=(-2, -1), keepdim=True) + 1e-6)

        phase = torch.angle(fft)
        phase = phase / torch.pi

        freq_input = torch.cat([magnitude, phase], dim=1)

        out = self.features(freq_input)
        out = self.classifier(out)
        return out


class CombinedDetector(nn.Module):
    def __init__(self):
        super().__init__()

        self.xception = timm.create_model('xception', pretrained=True, num_classes=0)
        xception_features = self.xception.num_features  # 2048

        self.freq_branch = FrequencyBranch()
        freq_features = 128

        self.classifier = nn.Sequential(
            nn.Linear(xception_features + freq_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 2)
        )

    def forward(self, x_xception, x_freq):
        spatial_features = self.xception(x_xception)
        freq_features = self.freq_branch(x_freq)
        combined = torch.cat([spatial_features, freq_features], dim=1)
        out = self.classifier(combined)
        return out





##Loading Target and Comparison model

In [5]:
combined_model = CombinedDetector().cuda()
combined_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_combinedwimpfftsplitaug.pth'))
combined_model.eval()
print("CombinedDetector (refined) loaded")
print(f"Trainable parameters: {sum(p.numel() for p in combined_model.parameters() if p.requires_grad):,}")


xception_model = timm.create_model('xception', pretrained=True, num_classes=2)
xception_model.load_state_dict(torch.load('/content/drive/MyDrive/best_model_baselinefix.pth'))
xception_model = xception_model.cuda()
xception_model.eval()
print("XceptionOnly loaded")

/usr/local/lib/python3.12/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name xception to current legacy_xception.
  model = create_fn(


Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-cadene/xception-43020ad28.pth" to /root/.cache/torch/hub/checkpoints/xception-43020ad28.pth
CombinedDetector (refined) loaded
Trainable parameters: 22,142,938
XceptionOnly loaded


##Defining attacks to be used against proxy model

In [6]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def fgsm_attack(model, images, labels, epsilon):
    images = images.clone().detach().to(device)
    labels = labels.clone().detach().to(device)
    images.requires_grad = True

    outputs = model(images)
    loss = F.cross_entropy(outputs, labels)

    model.zero_grad()
    loss.backward()

    gradient_sign = images.grad.data.sign()
    adversarial_images = images + epsilon * gradient_sign
    adversarial_images = torch.clamp(adversarial_images, -1, 1)

    return adversarial_images.detach()


def pgd_attack(model, images, labels, epsilon, alpha, num_steps):
    images = images.clone().detach().to(device)
    labels = labels.clone().detach().to(device)

    adv_images = images + torch.empty_like(images).uniform_(-epsilon, epsilon)
    adv_images = torch.clamp(adv_images, -1, 1)

    for _ in range(num_steps):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = F.cross_entropy(outputs, labels)

        model.zero_grad()
        loss.backward()

        gradient_sign = adv_images.grad.data.sign()
        adv_images = adv_images.detach() + alpha * gradient_sign

        perturbation = torch.clamp(adv_images - images, -epsilon, epsilon)
        adv_images = torch.clamp(images + perturbation, -1, 1)

    return adv_images.detach()



##Querying the Black-Box Target (CombinedDetector)

In [7]:
def black_box_query(model, images):
    model.eval()
    with torch.no_grad():
        images = images.to(device)
        preds = model(images, images).argmax(dim=1).cpu().numpy()  # dual-input call
    return preds


all_query_imgs = []
all_predicted_labels = []

for imgs, _ in train_loader:
    predicted_lbls = black_box_query(combined_model, imgs)
    all_query_imgs.append(imgs)
    all_predicted_labels.append(predicted_lbls)

query_images = torch.cat(all_query_imgs, dim=0)
predicted_labels = np.concatenate(all_predicted_labels)

print(f"Total queries made to black box model: {len(predicted_labels)}")

Total queries made to black box model: 11200


##Constructing Proxy Training Dataset

In [8]:
from torch.utils.data import TensorDataset


proxy_dataset = TensorDataset(query_images, torch.tensor(predicted_labels, dtype=torch.long))
proxy_loader = DataLoader(proxy_dataset, batch_size=32, shuffle=True)

##Initialise Xception Proxy Model

In [9]:

proxy_model_new = timm.create_model('xception', pretrained=True, num_classes=2).cuda()
print(f"Trainable parameters: {sum(p.numel() for p in proxy_model_new.parameters() if p.requires_grad):,}")


Trainable parameters: 20,811,050


##Training the Xception Proxy Model

Trains the Xception-based surrogate using the labels returned by querying the CombinedDetector.
Cross-entropy loss is then computed against the target model’s predicted labels rather than the original dataset ground truth, with the aim of approximating the target’s decision behaviour.

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(proxy_model_new.parameters(), lr=0.0001)
num_epochs = 10


for epoch in range(num_epochs): ## Train the surrogate on labels predicted by the black-box target
    proxy_model_new.train()
    running_loss, correct, total = 0, 0, 0
    for xb, yb in proxy_loader:
        xb, yb = xb.cuda(), yb.cuda()
        optimizer.zero_grad()
        outputs = proxy_model_new(xb)
        loss = criterion(outputs, yb)  ##loss based on stolen labels not ground truth
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(yb).sum().item()  ## Accuracy here measuring agreement with the queried target labels
        total += yb.size(0)
    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss/len(proxy_loader):.4f} | Acc: {100.*correct/total:.2f}%")

torch.save(proxy_model_new.state_dict(), '/content/drive/MyDrive/myproxy_xceptionnew.pth')

Epoch 1/10 | Loss: 0.4868 | Acc: 74.96%
Epoch 2/10 | Loss: 0.2004 | Acc: 92.08%
Epoch 3/10 | Loss: 0.0853 | Acc: 96.95%
Epoch 4/10 | Loss: 0.0571 | Acc: 98.09%
Epoch 5/10 | Loss: 0.0377 | Acc: 98.69%
Epoch 6/10 | Loss: 0.0349 | Acc: 98.87%
Epoch 7/10 | Loss: 0.0302 | Acc: 98.99%
Epoch 8/10 | Loss: 0.0236 | Acc: 99.20%
Epoch 9/10 | Loss: 0.0277 | Acc: 99.21%
Epoch 10/10 | Loss: 0.0221 | Acc: 99.29%


##Black-Box Transfer Attack Evaluation

FGSM and PGD adversarial examples are generated using the trained Xception surrogate and then transferred to the target model for evaluation. The CombinedDetector is the primary black-box target, as the surrogate was trained using labels obtained by querying it.

In [11]:
def evaluate_transfer(target_model, attack_fn, epsilons, dual_input=False, **attack_kwargs):
    target_model.eval()
    evasion_rates = []

    for eps in epsilons:
        fooled = 0
        total_fake = 0
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            fake_mask = (labels == 0)
            if fake_mask.sum() == 0:
                continue
            fake_images = images[fake_mask]
            fake_labels = labels[fake_mask]

            adv_images = attack_fn(proxy_model_new, fake_images, fake_labels, eps, **attack_kwargs)

            with torch.no_grad():
                if dual_input:
                    outputs = target_model(adv_images, adv_images)
                else:
                    outputs = target_model(adv_images)
                predicted = outputs.argmax(1)
                fooled += (predicted == 1).sum().item()
                total_fake += fake_labels.size(0)

        rate = 100 * fooled / total_fake
        evasion_rates.append(rate)
        print(f"  ε={eps:.4f} → Victim evasion rate (transferred): {rate:.1f}% ({fooled}/{total_fake})")

    return evasion_rates


EPSILONS_BB = [0.0, 0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.10]

print("FGSM transfer to CombinedDetector...")
combined_fgsm_transfer = evaluate_transfer(combined_model, fgsm_attack, EPSILONS_BB, dual_input=True)

print("\nFGSM transfer to XceptionOnly...")
xception_fgsm_transfer = evaluate_transfer(xception_model, fgsm_attack, EPSILONS_BB, dual_input=False)  ##see markdown below

print("\nPGD transfer to CombinedDetector...")
combined_pgd_transfer = evaluate_transfer(combined_model, pgd_attack, EPSILONS_BB, alpha=0.01, num_steps=20, dual_input=True)

print("\nPGD transfer to XceptionOnly...")
xception_pgd_transfer = evaluate_transfer(xception_model, pgd_attack, EPSILONS_BB, alpha=0.01, num_steps=20, dual_input=False)  ##see markdown below

FGSM transfer to CombinedDetector...
  ε=0.0000 → Victim evasion rate (transferred): 8.1% (98/1205)
  ε=0.0050 → Victim evasion rate (transferred): 39.0% (470/1205)
  ε=0.0100 → Victim evasion rate (transferred): 39.9% (481/1205)
  ε=0.0200 → Victim evasion rate (transferred): 39.5% (476/1205)
  ε=0.0300 → Victim evasion rate (transferred): 38.9% (469/1205)
  ε=0.0500 → Victim evasion rate (transferred): 35.9% (433/1205)
  ε=0.0800 → Victim evasion rate (transferred): 28.8% (347/1205)
  ε=0.1000 → Victim evasion rate (transferred): 24.1% (291/1205)

FGSM transfer to XceptionOnly...
  ε=0.0000 → Victim evasion rate (transferred): 1.4% (17/1205)
  ε=0.0050 → Victim evasion rate (transferred): 25.7% (310/1205)
  ε=0.0100 → Victim evasion rate (transferred): 32.7% (394/1205)
  ε=0.0200 → Victim evasion rate (transferred): 35.7% (430/1205)
  ε=0.0300 → Victim evasion rate (transferred): 36.3% (437/1205)
  ε=0.0500 → Victim evasion rate (transferred): 34.9% (421/1205)
  ε=0.0800 → Victim eva

##Note on XceptionNet Results

The XceptionNet results are only included as a secondary cross-model transfer comparison and should not be interpreted as an equivalent black-box surrogate attack against XceptionNet. Report focuses on the CombinedDetector as the primary black-box target